# 13 BioDose AI Bridge - Python

## Biochemistry question

How can a synthetic dose-response notebook prepare careful, reusable logic for a future BioDose AI learning app?

This notebook does not build an app yet. It organizes the core steps: load data, validate columns, summarize responses, visualize dose-response patterns, review data quality, rank synthetic candidates, and export a short markdown summary.


In [ ]:
import plotly.io as pio
pio.renderers.default = "iframe"


## 1. Setup

The notebook uses only synthetic datasets and local helper functions from this repository. The helper import path is adjusted so the notebook can run from the `notebooks/` folder.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.data_quality import flag_outliers_zscore, simple_data_quality_score


## 2. Load Scenario Data

We use one clean synthetic dataset and one dataset with suspicious replicate values. Comparing both helps separate a visual response pattern from data-quality review.


In [ ]:
SCENARIOS = {
    "clear": PROJECT_ROOT / "data" / "drug_response" / "drug_response_clear.csv",
    "outlier": PROJECT_ROOT / "data" / "drug_response" / "drug_response_outlier.csv",
}

scenario_tables = {name: pd.read_csv(path) for name, path in SCENARIOS.items()}
scenario_tables["clear"].head()


## 3. Validate Required Columns

A future app should check that uploaded or selected data has the expected columns before calculating summaries or drawing plots.


In [ ]:
REQUIRED_COLUMNS = [
    "sample_id",
    "drug_name",
    "concentration_uM",
    "replicate",
    "cell_viability_percent",
]


def validate_required_columns(df, required_cols):
    missing = [col for col in required_cols if col not in df.columns]
    return {"valid": len(missing) == 0, "missing_columns": missing}


validation_results = {
    name: validate_required_columns(df, REQUIRED_COLUMNS)
    for name, df in scenario_tables.items()
}
validation_results


## 4. Compute Summary Statistics

The summary table gives the mean response, spread, replicate count, and SEM for each compound and concentration.


In [ ]:
def summarize_dose_response(df):
    return (
        df.groupby(["drug_name", "concentration_uM"])
        .agg(
            mean_viability=("cell_viability_percent", "mean"),
            sd_viability=("cell_viability_percent", "std"),
            n=("cell_viability_percent", "count"),
        )
        .reset_index()
        .assign(sem_viability=lambda x: x["sd_viability"] / (x["n"] ** 0.5))
    )


summaries = {name: summarize_dose_response(df) for name, df in scenario_tables.items()}
summaries["clear"].head()


## 5. Create Dose-Response Plot

This Plotly chart is the main visual object that a future BioDose app could reuse. Zero concentration is shown with a small placeholder value so it can appear on a log-scale axis.


In [ ]:
def plot_dose_response(summary_df, title):
    plot_df = summary_df.copy()
    plot_df["plot_concentration"] = plot_df["concentration_uM"].replace(0, 0.001)
    fig = px.line(
        plot_df,
        x="plot_concentration",
        y="mean_viability",
        color="drug_name",
        markers=True,
        error_y="sem_viability",
        hover_data=["concentration_uM", "sd_viability", "n"],
        title=title,
    )
    fig.update_xaxes(type="log", title="Concentration (uM, log scale; 0 plotted as 0.001)")
    fig.update_yaxes(title="Mean Cell Viability (%)")
    return fig


fig = plot_dose_response(summaries["clear"], "Synthetic Dose-Response Summary - Clear Scenario")
# If this chart does not render in Jupyter, try: fig.show(renderer="browser")
fig.show(renderer="iframe")


## 6. Compute Data Quality Score

The score is intentionally simple. It is a teaching signal for review, not a regulatory or clinical quality assessment.


In [ ]:
def scenario_quality_report(df):
    score = simple_data_quality_score(df, REQUIRED_COLUMNS, min_rows=12)
    flagged = flag_outliers_zscore(
        df,
        ["drug_name", "concentration_uM"],
        "cell_viability_percent",
        threshold=1.5,
    )

    group_range = flagged.groupby(["drug_name", "concentration_uM"])["cell_viability_percent"].transform(
        lambda values: values.max() - values.min()
    )
    flagged["range_flag"] = group_range > 20
    flagged["review_flag"] = (flagged["qc_flag"] == "review") | flagged["range_flag"]

    review_count = int(flagged["review_flag"].sum())
    suspicious_groups = int(
        flagged.loc[flagged["range_flag"], ["drug_name", "concentration_uM"]]
        .drop_duplicates()
        .shape[0]
    )
    notes = list(score["notes"])
    if review_count > 0:
        notes.append(f"{review_count} replicate value(s) flagged for review.")
    if suspicious_groups > 0:
        notes.append(f"{suspicious_groups} compound/concentration group(s) have a large replicate range.")
    return {
        "score": score["score"],
        "review_count": review_count,
        "notes": notes,
    }


quality_reports = {name: scenario_quality_report(df) for name, df in scenario_tables.items()}
quality_reports


## 7. Generate Candidate Ranking

This simple ranking looks at the highest tested concentration in each scenario. Lower mean viability at the highest concentration is ranked first, but the quality notes must be read before interpretation.


In [ ]:
def rank_candidates(summary_df, scenario_name):
    max_concentration = summary_df["concentration_uM"].max()
    ranked = (
        summary_df[summary_df["concentration_uM"] == max_concentration]
        .sort_values("mean_viability")
        .loc[:, ["drug_name", "concentration_uM", "mean_viability", "sem_viability", "n"]]
        .reset_index(drop=True)
    )
    ranked.insert(0, "scenario", scenario_name)
    ranked.insert(1, "rank", range(1, len(ranked) + 1))
    return ranked


rankings = pd.concat(
    [rank_candidates(summary, name) for name, summary in summaries.items()],
    ignore_index=True,
)
rankings


## 8. Generate Challenge Questions

These questions can be shown in a future app or used as notebook reflection prompts.


In [ ]:
challenge_questions = [
    "Which synthetic compound has the lowest mean viability at the highest tested concentration?",
    "Does the outlier scenario change how confident you feel about the visual pattern?",
    "Which replicate values should be reviewed before writing an interpretation?",
    "What limitation should be included in a figure caption for this synthetic dataset?",
]

for question in challenge_questions:
    print("-", question)


## 9. Export Summary Markdown

When this cell runs, it writes a small markdown report to `outputs/biodose_bridge_summary.md`. The `outputs/` folder is local working output and is intentionally not part of the main teaching data.


In [ ]:
def make_markdown_summary(rankings, quality_reports):
    lines = [
        "# BioDose AI Bridge Summary",
        "",
        "This summary uses synthetic dose-response datasets for learning only.",
        "It is not clinical, diagnostic, regulatory, or drug-efficacy evidence.",
        "",
        "## Candidate Ranking at Highest Tested Concentration",
        "",
        rankings.to_markdown(index=False),
        "",
        "## Data Quality Notes",
        "",
    ]
    for scenario, report in quality_reports.items():
        lines.append(f"### {scenario}")
        lines.append(f"- Score: {report['score']}/100")
        lines.append(f"- Review flags: {report['review_count']}")
        if report["notes"]:
            for note in report["notes"]:
                lines.append(f"- {note}")
        else:
            lines.append("- No major notes from the simple quality check.")
        lines.append("")
    lines.extend([
        "## Suggested Next Step",
        "",
        "Review flagged replicate values, compare the clear and outlier scenarios, and write a cautious figure caption.",
    ])
    return "\n".join(lines)


output_dir = PROJECT_ROOT / "outputs"
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "biodose_bridge_summary.md"
output_path.write_text(make_markdown_summary(rankings, quality_reports), encoding="utf-8")
output_path


## Interpretation Practice

- Which candidate ranks first in each synthetic scenario?
- How do the data quality notes affect your confidence in the ranking?
- Which plot or table would be most useful in a future BioDose app?
- What warning should always appear with synthetic dose-response outputs?

## Limitations

- These datasets are synthetic and designed for learning.
- Candidate ranking is based on a simple teaching rule, not a real screening decision.
- The output should not be described as clinical, diagnostic, regulatory, or drug-efficacy evidence.
- A future app would need stronger validation, clearer user guidance, and careful review of any uploaded data.
